In [13]:
import sys
print(sys.executable)

D:\condaData\envs_dirs\env_name\python.exe


In [14]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

def get_weather(city: str) -> str:
    """
    查询指定城市的实时天气（使用 OpenWeatherMap API）
    
    参数:
        city: 城市名（中文或英文）
    
    返回:
        天气信息字符串
    """
    api_key = os.getenv("OPENWEATHER_API_KEY")  # 从环境变量读取
    if not api_key:
        return "错误：未配置天气 API 密钥"
    
    # 支持中文城市名，但 API 需要拼音或英文，这里先尝试直接传参
    url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric&lang=zh_cn"
    
    try:
        response = requests.get(url, timeout=5)
        data = response.json()
        if response.status_code != 200:
            return f"查询失败：{data.get('message', '未知错误')}"
        
        weather_desc = data["weather"][0]["description"]
        temp = data["main"]["temp"]
        humidity = data["main"]["humidity"]
        return f"{city} 天气：{weather_desc}，温度 {temp}℃，湿度 {humidity}%"
    except Exception as e:
        return f"天气查询出错：{e}"

In [15]:
from langchain_core.tools import tool

@tool
def weather_tool(city: str) -> str:
    """
    查询指定城市的实时天气。
    
    输入格式：城市名称（例如：北京、上海）
    """
    return get_weather(city)

In [16]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# 复用之前的嵌入模型（硅基流动）
embeddings = OpenAIEmbeddings(
    model="BAAI/bge-large-zh-v1.5",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1"
)

# 加载持久化的向量库
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

In [18]:
def search_library(query: str, k: int = 3) -> str:
    """
    在图书馆文档中检索与问题相关的信息。
    
    参数:
        query: 用户的问题
        k: 返回的文档块数量
    
    返回:
        检索到的内容（多个块拼接）
    """
    docs = vectorstore.similarity_search(query, k=k)
    if not docs:
        return "未找到相关信息。"
    context = "\n\n".join([doc.page_content for doc in docs])
    return context

In [19]:
@tool
def rag_search_tool(query: str) -> str:
    """
    在图书馆知识库中检索信息。当用户询问图书馆相关规则、开放时间、借阅等问题时使用。
    """
    return search_library(query, k=3)

In [8]:
# 之前定义的工具
from tools import add_tool, multiply_tool  # 假设已导入

# 新工具
tools = [add_tool, multiply_tool, weather_tool, rag_search_tool]


向量库加载成功


In [20]:
from langchain_core.prompts import ChatPromptTemplate

react_prompt_with_memory = ChatPromptTemplate.from_template("""
Answer the following questions as best you can. You have access to the following tools:

{tools}

Previous conversation:
{chat_history}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought: {agent_scratchpad}
""")

In [21]:
from langchain.agents import create_react_agent, AgentExecutor
from langchain.memory import ConversationBufferMemory
from langchain_openai import ChatOpenAI

# 配置 LLM（使用硅基流动）
llm = ChatOpenAI(
    model="deepseek-ai/DeepSeek-V3",
    openai_api_key=os.getenv("siliconFlow_API_KEY"),
    openai_api_base="https://api.siliconflow.cn/v1",
    temperature=0.3
)

# 记忆
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Agent
agent = create_react_agent(llm, tools, react_prompt_with_memory)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    max_iterations=5,
    handle_parsing_errors=True
)

In [22]:
# 测试1：知识库问答（图书馆开放时间）
print("=== 测试1：知识库问答 ===")
res1 = agent_executor.invoke({"input": "方闻馆开放时间是什么？"})
print("回答:", res1["output"])
"""
# 测试2：实时天气
print("\n=== 测试2：实时天气 ===")
res2 = agent_executor.invoke({"input": "今天上海天气怎么样？"})
print("回答:", res2["output"])

# 测试3：混合任务（需要记忆+工具+检索）
print("\n=== 测试3：混合任务 ===")
# 先记住名字
agent_executor.invoke({"input": "我叫小明"})
# 然后问复杂问题
res3 = agent_executor.invoke({"input": "我叫什么名字？方闻馆几点关门？今天上海天气如何？帮我算一下 15 + 27"})
print("回答:", res3["output"])
"""

=== 测试1：知识库问答 ===


> Entering new AgentExecutor chain...
用户询问的是图书馆的开放时间，这属于图书馆相关规则的问题，应该使用rag_search_tool来检索信息。

Action: rag_search_tool
Action Input: 方闻馆开放时间一、借阅权限
读者类型              外借册数    预约册数    预约催还过期停借册数
─────────────────────────────────────────────────────────────
在校教职工            100册       30册        3册
全日制本科生          100册       30册        3册
全日制研究生          100册       30册        3册
非全日制研究生        30册        3册         1册

企编教工              30册        3册         1册
访问学者              30册        3册         1册
离退休人员            5册         3册         1册
校内其他人员          5册         1册         1册
继续教育读者          5册         1册         1册

三、方闻馆详细时间
─────────────────────────────────
咨询台：周一至周日 8:30-22:30
东阅览室：周一至周日 8:30-22:30
西阅览室：周一至周日 8:30-22:30
珍本阅览室：周一至周五 08:30-12:00；13:30-17:30（周六日不开放）
三层密集书库：周一至周五 08:30-12:00；13:30-17:30（周六日不开放）Final Answer: 方闻馆的开放时间如下：
- 咨询台：周一至周日 8:30-22:30
- 东阅览室：周一至周日 8:30-22:30
- 西阅览室：周一至周日 8:30-22:30
- 珍本阅览室：周一至周五 08:30-12:00；13:30-17:30（周六日不开放）
- 三层密集书库：周一

'\n# 测试2：实时天气\nprint("\n=== 测试2：实时天气 ===")\nres2 = agent_executor.invoke({"input": "今天上海天气怎么样？"})\nprint("回答:", res2["output"])\n\n# 测试3：混合任务（需要记忆+工具+检索）\nprint("\n=== 测试3：混合任务 ===")\n# 先记住名字\nagent_executor.invoke({"input": "我叫小明"})\n# 然后问复杂问题\nres3 = agent_executor.invoke({"input": "我叫什么名字？方闻馆几点关门？今天上海天气如何？帮我算一下 15 + 27"})\nprint("回答:", res3["output"])\n'

In [11]:
# 测试1：知识库问答（图书馆开放时间）
print("=== 测试1：知识库问答 ===")
res1 = agent_executor.invoke({"input": "方闻馆开放时间是什么？"})
print("回答:", res1["output"])

=== 测试1：知识库问答 ===


> Entering new AgentExecutor chain...
用户询问的是方闻馆的开放时间，这属于图书馆相关规则的问题，应该使用rag_search_tool进行检索。

Action: rag_search_tool  
Action Input: 方闻馆开放时间  一、借阅权限
读者类型              外借册数    预约册数    预约催还过期停借册数
─────────────────────────────────────────────────────────────
在校教职工            100册       30册        3册
全日制本科生          100册       30册        3册
全日制研究生          100册       30册        3册
非全日制研究生        30册        3册         1册

企编教工              30册        3册         1册
访问学者              30册        3册         1册
离退休人员            5册         3册         1册
校内其他人员          5册         1册         1册
继续教育读者          5册         1册         1册

三、方闻馆详细时间
─────────────────────────────────
咨询台：周一至周日 8:30-22:30
东阅览室：周一至周日 8:30-22:30
西阅览室：周一至周日 8:30-22:30
珍本阅览室：周一至周五 08:30-12:00；13:30-17:30（周六日不开放）
三层密集书库：周一至周五 08:30-12:00；13:30-17:30（周六日不开放）Final Answer: 方闻馆各区域开放时间如下：
- 咨询台：周一至周日 8:30-22:30  
- 东阅览室：周一至周日 8:30-22:30  
- 西阅览室：周一至周日 8:30-22:30  
- 珍本阅览室：周一至周五 08:30-12:00；13:30-17:30（周六日不开放） 